In [1]:
#import module & create spark session

from pyspark.sql import SparkSession
import pyspark.sql.types as types
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('module6_homework') \
    .getOrCreate()

print(f"Spark version: {spark.version}")   

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 22:29:02 WARN Utils: Your hostname, JImi-Dell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/11 22:29:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 22:29:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/11 22:29:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


In [2]:
#download homework data

!wget -O yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget -O taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-11 22:29:07--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.99.156, 18.172.99.3, 18.172.99.89, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.99.156|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  46.7MB/s    in 1.5s    

2026-03-11 22:29:08 (46.7 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]

--2026-03-11 22:29:08--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.99.122, 18.172.99.89, 18.172.99.3, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.99.122|:443... connected.
HTTP request sent, awaiting resp

In [3]:
# Check files downloaded
!ls -lh *.parquet *.csv

-rw-r--r-- 1 olujimim olujimim 295M Jun 30  2022 fhvhv_tripdata_2021-01.parquet
-rw-r--r-- 1 olujimim olujimim 262K Mar  9 08:44 head.csv
-rw-r--r-- 1 olujimim olujimim  13K Feb 22  2024 taxi_zone_lookup.csv
-rw-r--r-- 1 olujimim olujimim  68M Dec 19 15:51 yellow_tripdata_2025-11.parquet


In [5]:
#read the Parquet file Without a schema first to see what is inside
df_raw = spark.read.parquet('yellow_tripdata_2025-11.parquet')

#view structure: volumn names and types
df_raw.printSchema()

#show first 5 rows of actual data
df_raw.show(5)

#count total rows
print(f"\n TOTAL ROWS: {df_raw.count():,}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [6]:
#Question 1

In [7]:
print(spark.version)

4.1.1


In [8]:
#Question 2
#Average size of the Parquet in MB

In [9]:
#Repartition to 4
df_repartitioned = df_raw.repartition(4)

#save to a new folder
df_repartitioned.write.parquet('yellow_tripdata_2025-11_4partitions', mode='overwrite')

In [10]:
#check file sizes

import os
import glob

#find all parquet files in folder
parquet_files = glob.glob('yellow_tripdata_2025-11_4partitions/part-*.parquet')

#calculate average size
total_size = 0
for file in parquet_files:
    size_mb = os.path.getsize(file) / (1024 * 1024)  #converts bytes to MB
    print(f"{os.path.basename(file)}: {size_mb: .2f} MB")
    total_size += size_mb

#get avg_size
avg_size = total_size / len(parquet_files)
print(f"\n Average File Size: {avg_size: .1f} MB")

part-00000-927c8bd1-4d4e-44b5-b886-bf3f41941b60-c000.snappy.parquet:  24.41 MB
part-00003-927c8bd1-4d4e-44b5-b886-bf3f41941b60-c000.snappy.parquet:  24.42 MB
part-00001-927c8bd1-4d4e-44b5-b886-bf3f41941b60-c000.snappy.parquet:  24.41 MB
part-00002-927c8bd1-4d4e-44b5-b886-bf3f41941b60-c000.snappy.parquet:  24.42 MB

 Average File Size:  24.4 MB


In [13]:
#Question 3: Count records
#How many trips were there on thd 15th November?

In [17]:
#using date filter
from pyspark.sql import functions as F

#create a date column for easier filtering
df_with_date = df_raw.withColumn(
    'pickup_date',
    F.to_date('tpep_pickup_datetime') #convert timestamp to date
)

#count trips where pickup date is Nov 15, 2025
Nov15_trip_count = df_with_date.filter(F.col('pickup_date') == '2025-11-15').count()

#print count
print(f"Trips on Nov. 15, 2025: {Nov15_trip_count: ,}")

Trips on Nov. 15, 2025:  162,604


In [11]:
#Question 4: Longest trip

In [19]:
#calculate trip duration in hours
df_duration = df_raw.withColumn(
    'trip_duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

#find max duration
max_duration = df_duration.agg(F.max('trip_duration_hours')).collect()[0][0]

print(f"Longest trip: {max_duration: .1f} hours")

Longest trip:  90.6 hours


In [20]:
#Question 6. Least frequent pickup location zone 

In [22]:
#read zone lookup data
df_zones = spark.read.option('header', 'true').csv('taxi_zone_lookup.csv')

#view data
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [31]:
#count trips per pickkup location ID
pickup_counts = df_raw.groupBy('PULocationID').count()
#Trip count per location ID (first 5)

pickup_counts.show(5)

+------------+-----+
|PULocationID|count|
+------------+-----+
|         148|51711|
|         243| 4901|
|          31|   89|
|         137|46493|
|          85| 1711|
+------------+-----+
only showing top 5 rows


In [40]:
#Join with zones
df_with_zone_names = pickup_counts.join(
    df_zones,
    pickup_counts.PULocationID == df_zones.LocationID,
    'inner'
).select('Zone', 'count')

#find least frequent zones (samllest count)
least_frequent = df_with_zone_names.orderBy('count').first()

print(f"Least frequent pickup zone: {least_frequent['Zone']}")

print(f"\n (only {least_frequent['count']} trips")


#Check all options to see which is correct
options = [
    "Governor's Island/Ellis Island/Liberty Island",
    "Arden Heights",
    "Rikers Island",
    "Jamaica Bay"
]


print("\n Checking options: ")
for zones in options:
    count = df_with_zone_names.filter(F.col('Zone') == zones).select('count').collect()

    if count:
        print(f"{zones}: {count[0]['count']} trips")
    else:
        print(f"{zones}: 0 trips (not found)")



#Double-check which one has the smallest number
print("\n All Zones Sorted By Count (smallest first):")
df_with_zone_names.orderBy('count').show(15, truncate=False)

Least frequent pickup zone: Governor's Island/Ellis Island/Liberty Island

 (only 1 trips

 Checking options: 
Governor's Island/Ellis Island/Liberty Island: 1 trips
Arden Heights: 1 trips
Rikers Island: 4 trips
Jamaica Bay: 5 trips

 All Zones Sorted By Count (smallest first):
+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Arden Heights                                |1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Port Richmond                                |3    |
|Rossville/Woodrow                            |4    |
|Rikers Island                                |4    |
|Green-Wood Cemetery                          |4    |
|Great Kills                                  |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
|Oakwood           